# Notebook 06 — LM Variants Evaluation

Three targeted improvements to the base Levenberg-Marquardt optimizer, plus a three-phase hybrid, motivated by lessons learned from the Proximal optimizer:

| Variant | Inspiration | Key change |
|---|---|---|
| **LM_base** | — | Original: Υ₂ from iter 0, stop on MSE only |
| **LM_delayed** | Proximal Phase 1 | Suppress Υₙ for first 30% of iterations |
| **LM_progressive** | Proximal λ warm-up | n in Υₙ: 2→4→8→16 over training |
| **LM_dual** | Proximal dual stopping | Stop only when MSE **and** Δ(N)/P < tol_dn |
| **LM_hybrid** | Proximal full pipeline | Phase 1: LM → Phase 2: Adam + ternary reg (λ warm-up) → Phase 3: hardening (3× reg, 200 steps) |

Evaluated on: **MONK-1** (17 features, 124 train, clean logical rule) and **Mushroom** (111 features, high fan-in).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from luknn.network.luknn import make_network
from luknn.benchmark.datasets import load_monk, load_mushroom
from luknn.benchmark.metrics import compute_f1, compute_accuracy, compute_delta_n
from luknn.network.crystallization import (
    representation_error, progressive_crystallize,
    crisp_crystallize_weights, crisp_crystallize_bias,
)
from luknn.optimizers import (
    LMOptimizer, LMDelayedOptimizer, LMProgressiveOptimizer, LMDualOptimizer,
    LMHybridOptimizer,
)
import torch.nn as nn

print('Setup OK')

## 1 · Parameters

In [ ]:
BASE_SEED   = 42
N_TRIALS    = 10        # trials per variant per dataset
MAX_ITER    = 2000      # generous budget — LM steps are cheap on MONK
TOL_MSE     = 2e-3
BATCH_SIZE_MUSHROOM = 256   # mini-batch Jacobian for 8124 rows

RESULTS_DIR = '../results/lm_variants'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Variant definitions: name → factory(model)
VARIANT_DEFS = {
    'LM_base': lambda m: LMOptimizer(
        m, mu_init=1e-2, crystallize_n=2, patience=50, prune=False),
    'LM_delayed': lambda m: LMDelayedOptimizer(
        m, mu_init=1e-2, crystallize_n=2,
        crystallize_start_fraction=0.3, patience=50, prune=False),
    'LM_progressive': lambda m: LMProgressiveOptimizer(
        m, mu_init=1e-2,
        n_schedule=(2, 4, 8, 16),
        schedule_fractions=(0.0, 0.5, 0.75, 0.9),
        patience=50, prune=False),
    'LM_dual': lambda m: LMDualOptimizer(
        m, mu_init=1e-2, crystallize_n=2,
        tol_dn=0.05, dn_patience=50, patience=50, prune=False),
    'LM_hybrid': lambda m: LMHybridOptimizer(
        m, mu_init=1e-2, crystallize_n=2,
        p1_fraction=0.4, p1_patience=30,
        lr_p2=1e-2, lambda_sparse=1e-3, lambda_attract=0.1,
        prox_threshold=5e-4, tol_dn=0.05, p2_patience=50,
        p3_steps=200,   # Phase 3: 200 fixed steps at 3× reg (hardening)
        prune=False),
}

VARIANT_COLORS = {
    'LM_base':        'steelblue',
    'LM_delayed':     'darkorange',
    'LM_progressive': 'seagreen',
    'LM_dual':        'crimson',
    'LM_hybrid':      'purple',
}

print('Parameters set.')

## 2 · Helpers

In [ ]:
def force_crystallize(model):
    """Crisp crystallization regardless of convergence state."""
    for m in model.modules():
        if hasattr(m, 'crystallize'):
            m.crystallize()
        elif isinstance(m, nn.Linear):
            m.weight.data = crisp_crystallize_weights(progressive_crystallize(m.weight.data))
            m.bias.data   = crisp_crystallize_bias(progressive_crystallize(m.bias.data))


def run_trial(variant_name, make_opt, ds, seed, max_iter, tol_mse, batch_size=0):
    torch.manual_seed(seed)
    model = make_network(ds.n_features, n_hidden_layers=2, hidden_width=ds.n_features)
    opt   = make_opt(model)
    if hasattr(opt, 'batch_size'):
        opt.batch_size = batch_size

    res = opt.train(ds.X_train, ds.y_train, tol_mse=tol_mse, max_iter=max_iter)

    # Pre-crystallization continuous F1
    with torch.no_grad():
        pred_cont = model(ds.X_test)
    f1_cont = compute_f1(pred_cont, ds.y_test)
    dn_pre  = compute_delta_n(model)

    # Force crystallization, measure symbolic quality
    force_crystallize(model)
    with torch.no_grad():
        pred_crisp = model(ds.X_test)
    f1_crisp = compute_f1(pred_crisp, ds.y_test)
    dn_post  = compute_delta_n(model)
    cryst    = dn_post < 1e-2

    return {
        'variant': variant_name, 'seed': seed,
        'f1_cont':  round(f1_cont, 4),
        'f1_crisp': round(f1_crisp, 4),
        'dn_pre':   round(dn_pre, 4),
        'dn_post':  round(dn_post, 4),
        'crystallized': cryst,
        'converged':    res.converged,
        'iterations':   res.iterations,
        'final_mse':    round(res.final_mse, 6),
        'time_s':       round(res.total_time_s, 2),
        'reason':       res.reason,
        'mse_history':  res.mse_history,
    }

## 3 · MONK-1 benchmark

In [ ]:
MONK_CSV = f'{RESULTS_DIR}/monk_variants_p3.csv'

if os.path.exists(MONK_CSV):
    monk_df = pd.read_csv(MONK_CSV)
    print(f'Loaded pre-computed MONK results from {MONK_CSV}')
else:
    ds_monk = load_monk(problem=1, seed=BASE_SEED)
    print(f'MONK-1: features={ds_monk.n_features}  train={len(ds_monk.X_train)}  test={len(ds_monk.X_test)}')

    monk_rows = []
    for vname, make_opt in VARIANT_DEFS.items():
        print(f'\n[{vname}]')
        for trial in range(N_TRIALS):
            seed = BASE_SEED + trial * 17
            r = run_trial(vname, make_opt, ds_monk, seed, MAX_ITER, TOL_MSE)
            row = {k: v for k, v in r.items() if k != 'mse_history'}
            monk_rows.append(row)
            print(f'  trial {trial:2d}: F1_cont={r["f1_cont"]:.3f}  '
                  f'F1_crisp={r["f1_crisp"]:.3f}  cryst={r["crystallized"]}  '
                  f'ΔN={r["dn_pre"]:.0f}  iters={r["iterations"]:4d}  '
                  f'{r["reason"]}')

    monk_df = pd.DataFrame(monk_rows)
    monk_df.to_csv(MONK_CSV, index=False)
    print('\nDone.')

In [ ]:
agg = monk_df.groupby('variant').agg(
    f1_cont_mean  = ('f1_cont',  'mean'),
    f1_cont_std   = ('f1_cont',  'std'),
    f1_crisp_mean = ('f1_crisp', 'mean'),
    f1_crisp_std  = ('f1_crisp', 'std'),
    cryst_rate    = ('crystallized', 'mean'),
    conv_rate     = ('converged',    'mean'),
    dn_pre_mean   = ('dn_pre',  'mean'),
    iters_mean    = ('iterations', 'mean'),
    time_mean     = ('time_s',  'mean'),
).round(3)

# reindex so table follows variant order
agg = agg.reindex(list(VARIANT_DEFS.keys()))
print('MONK-1 Summary')
agg

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MONK-1 — LM Variants Comparison', fontsize=13)

order = list(VARIANT_DEFS.keys())
colors = [VARIANT_COLORS[v] for v in order]

# F1 continuous
ax = axes[0]
data_cont = [monk_df[monk_df['variant'] == v]['f1_cont'].values for v in order]
bp = ax.boxplot(data_cont, patch_artist=True, labels=order)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
ax.set_title('F1 (continuous, pre-crystallization)')
ax.set_ylabel('F1')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=15)

# F1 crisp
ax = axes[1]
data_crisp = [monk_df[monk_df['variant'] == v]['f1_crisp'].values for v in order]
bp = ax.boxplot(data_crisp, patch_artist=True, labels=order)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
ax.set_title('F1 (crisp, after forced crystallization)')
ax.set_ylabel('F1')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=15)

# Δ(N) pre-crystallization
ax = axes[2]
data_dn = [monk_df[monk_df['variant'] == v]['dn_pre'].values for v in order]
bp = ax.boxplot(data_dn, patch_artist=True, labels=order)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
ax.set_title('Δ(N) before forced crystallization\n(lower = closer to integers)')
ax.set_ylabel('Δ(N)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/monk_variants_boxplot.png', dpi=150)
plt.show()

In [ ]:
# Re-run a single representative trial per variant to capture MSE curves
ds_monk = load_monk(problem=1, seed=BASE_SEED)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('MONK-1 — MSE learning curves (seed=42)', fontsize=12)
axes = axes.flatten()

for ax, (vname, make_opt) in zip(axes, VARIANT_DEFS.items()):
    torch.manual_seed(BASE_SEED)
    model = make_network(ds_monk.n_features, n_hidden_layers=2, hidden_width=ds_monk.n_features)
    opt   = make_opt(model)
    res   = opt.train(ds_monk.X_train, ds_monk.y_train, tol_mse=TOL_MSE, max_iter=MAX_ITER)
    ax.semilogy(res.mse_history, color=VARIANT_COLORS[vname], lw=1.5)
    ax.axhline(TOL_MSE, color='grey', ls='--', lw=1, label=f'tol_mse={TOL_MSE}')
    ax.set_title(vname)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('MSE (log scale)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/monk_variants_curves.png', dpi=150)
plt.show()

## 4 · Mushroom benchmark (high fan-in: 111 features)

In [ ]:
MUSH_CSV = f'{RESULTS_DIR}/mushroom_variants_p3.csv'

if os.path.exists(MUSH_CSV):
    mush_df = pd.read_csv(MUSH_CSV)
    print(f'Loaded pre-computed Mushroom results from {MUSH_CSV}')
else:
    ds_mush = load_mushroom(seed=BASE_SEED)
    print(f'Mushroom: features={ds_mush.n_features}  train={len(ds_mush.X_train)}  '
          f'test={len(ds_mush.X_test)}')
    print(f'Using mini-batch Jacobian (batch_size={BATCH_SIZE_MUSHROOM})')

    mush_rows = []
    for vname, make_opt in VARIANT_DEFS.items():
        print(f'\n[{vname}]')
        for trial in range(5):  # 5 trials on mushroom (slower)
            seed = BASE_SEED + trial * 17
            r = run_trial(vname, make_opt, ds_mush, seed,
                          max_iter=1000, tol_mse=TOL_MSE,
                          batch_size=BATCH_SIZE_MUSHROOM)
            row = {k: v for k, v in r.items() if k != 'mse_history'}
            mush_rows.append(row)
            print(f'  trial {trial:2d}: F1_cont={r["f1_cont"]:.3f}  '
                  f'F1_crisp={r["f1_crisp"]:.3f}  cryst={r["crystallized"]}  '
                  f'ΔN={r["dn_pre"]:.0f}  iters={r["iterations"]:4d}  '
                  f'{r["reason"]}')

    mush_df = pd.DataFrame(mush_rows)
    mush_df.to_csv(MUSH_CSV, index=False)
    print('\nDone.')

In [ ]:
agg_mush = mush_df.groupby('variant').agg(
    f1_cont_mean  = ('f1_cont',  'mean'),
    f1_cont_std   = ('f1_cont',  'std'),
    f1_crisp_mean = ('f1_crisp', 'mean'),
    f1_crisp_std  = ('f1_crisp', 'std'),
    cryst_rate    = ('crystallized', 'mean'),
    conv_rate     = ('converged',    'mean'),
    dn_pre_mean   = ('dn_pre',  'mean'),
    iters_mean    = ('iterations', 'mean'),
    time_mean     = ('time_s',  'mean'),
).round(3)

agg_mush = agg_mush.reindex(list(VARIANT_DEFS.keys()))
print('Mushroom Summary')
agg_mush

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Mushroom — LM Variants Comparison (high fan-in)', fontsize=13)

for ax, col_name, title in zip(axes,
    ['f1_cont', 'f1_crisp', 'dn_pre'],
    ['F1 (continuous)', 'F1 (crisp)', 'Δ(N) pre-crystallization']):
    data = [mush_df[mush_df['variant'] == v][col_name].values for v in order]
    bp = ax.boxplot(data, patch_artist=True, labels=order)
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.6)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
    if 'f1' in col_name:
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('F1')
    else:
        ax.set_ylabel('Δ(N)')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mushroom_variants_boxplot.png', dpi=150)
plt.show()

## 5 · Joint comparison — F1 crisp across datasets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (ds_label, df_) in zip(axes, [('MONK-1', monk_df), ('Mushroom', mush_df)]):
    x = np.arange(len(order))
    means = [df_[df_['variant'] == v]['f1_crisp'].mean() for v in order]
    stds  = [df_[df_['variant'] == v]['f1_crisp'].std()  for v in order]
    bars  = ax.bar(x, means, yerr=stds, capsize=5,
                   color=[VARIANT_COLORS[v] for v in order], alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=12, ha='right')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('F1 (post-crystallization)')
    ax.set_title(f'{ds_label} — F1 crisp (mean ± std)')
    ax.axhline(0, color='grey', lw=0.5)
    ax.grid(axis='y', alpha=0.3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.02,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/joint_f1_crisp.png', dpi=150)
plt.show()

## 6 · Δ(N) analysis — do variants leave weights closer to integers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Δ(N) before forced crystallization — lower is better', fontsize=12)

for ax, (ds_label, df_) in zip(axes, [('MONK-1', monk_df), ('Mushroom', mush_df)]):
    x = np.arange(len(order))
    means = [df_[df_['variant'] == v]['dn_pre'].mean() for v in order]
    stds  = [df_[df_['variant'] == v]['dn_pre'].std()  for v in order]
    ax.bar(x, means, yerr=stds, capsize=5,
           color=[VARIANT_COLORS[v] for v in order], alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=12, ha='right')
    ax.set_ylabel('Δ(N) = Σ(wᵢ − floor(wᵢ))')
    ax.set_title(ds_label)
    ax.grid(axis='y', alpha=0.3)
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 1, f'{m:.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dn_pre_comparison.png', dpi=150)
plt.show()

## 7 · Crystallization rate and convergence rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Crystallization & convergence rates', fontsize=12)

for ax, (ds_label, df_) in zip(axes, [('MONK-1', monk_df), ('Mushroom', mush_df)]):
    x = np.arange(len(order))
    w = 0.35
    cryst_rates = [df_[df_['variant'] == v]['crystallized'].mean() for v in order]
    conv_rates  = [df_[df_['variant'] == v]['converged'].mean()     for v in order]
    ax.bar(x - w/2, cryst_rates, width=w, label='Cryst. rate (after forced cryst.)',
           color=[VARIANT_COLORS[v] for v in order], alpha=0.8)
    ax.bar(x + w/2, conv_rates, width=w, label='Conv. rate (mse < tol_mse)',
           color=[VARIANT_COLORS[v] for v in order], alpha=0.4, hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=12, ha='right')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Rate')
    ax.set_title(ds_label)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cryst_conv_rates.png', dpi=150)
plt.show()

## 8 · Conclusions

Summarise findings from the cells above here once results are available.

In [ ]:
print('=== MONK-1 ===')
for v in order:
    g = monk_df[monk_df['variant'] == v]
    print(f'  {v:<20}  F1_crisp={g["f1_crisp"].mean():.3f}±{g["f1_crisp"].std():.3f}  '
          f'cryst={g["crystallized"].mean():.0%}  ΔN={g["dn_pre"].mean():.0f}  '
          f'conv={g["converged"].mean():.0%}')

print()
print('=== Mushroom ===')
for v in order:
    g = mush_df[mush_df['variant'] == v]
    print(f'  {v:<20}  F1_crisp={g["f1_crisp"].mean():.3f}±{g["f1_crisp"].std():.3f}  '
          f'cryst={g["crystallized"].mean():.0%}  ΔN={g["dn_pre"].mean():.0f}  '
          f'conv={g["converged"].mean():.0%}')